# Daily Challenge: Stock Price Prediction with LSTM

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import pickle
import os

In [3]:
# --- Config: change these to try different stocks or ETFs ---
SYMBOL   = "AAPL"       # any symbol from your stocks/ or etfs/ folder
FOLDER   = "stocks"     # "stocks" or "etfs"
FEATURES = ["Close"]    # columns to use as input features

# Load the symbol's CSV file
path = os.path.join(FOLDER, f"{SYMBOL}.csv")
df = pd.read_csv(path, parse_dates=["Date"], index_col="Date")
df = df[FEATURES].dropna()

# Create target: next day's closing price
df["Target"] = df["Close"].shift(-1)
df = df.dropna()

# Normalize all columns between 0 and 1
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df)

print(f"Loaded {SYMBOL} | rows: {len(df)} | columns: {list(df.columns)}")
df.tail()

Loaded AAPL | rows: 9908 | columns: ['Close', 'Target']


,Close,Target
Date,,
2020-03-25,245.520004,258.440002
2020-03-26,258.440002,247.740005
2020-03-27,247.740005,254.809998
2020-03-30,254.809998,254.289993
2020-03-31,254.289993,240.910004


In [4]:
# Split: 70% train | 15% val | 15% test
n         = len(scaled)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X = scaled[:, :-1].reshape(-1, 1, len(FEATURES))  # (samples, seq_len, features)
y = scaled[:, -1]                                     # target (next close)

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(StockDataset(X[:train_end],        y[:train_end]),        batch_size=32, shuffle=True)
val_loader   = DataLoader(StockDataset(X[train_end:val_end], y[train_end:val_end]), batch_size=32)
test_loader  = DataLoader(StockDataset(X[val_end:],          y[val_end:]),          batch_size=32)

print(f"Train: {train_end} | Val: {val_end-train_end} | Test: {n-val_end}")

Train: 6935 | Val: 1486 | Test: 1487


In [5]:
class StockGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = self.dropout(out[:, -1, :])
        return self.fc(out).squeeze(-1)

model = StockGRU(input_size=len(FEATURES))
print(model)

StockGRU(
  (gru): GRU(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
EPOCHS    = 30

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            val_loss += criterion(model(X_batch), y_batch).item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss/len(train_loader):.4f} | Val: {val_loss/len(val_loader):.4f}")

Epoch 5/30 | Train: 0.0000 | Val: 0.0000
Epoch 10/30 | Train: 0.0000 | Val: 0.0000
Epoch 15/30 | Train: 0.0000 | Val: 0.0000
Epoch 20/30 | Train: 0.0000 | Val: 0.0000
Epoch 25/30 | Train: 0.0000 | Val: 0.0001
Epoch 30/30 | Train: 0.0000 | Val: 0.0001


In [7]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        all_preds.extend(model(X_batch).numpy())
        all_targets.extend(y_batch.numpy())

r2 = r2_score(all_targets, all_preds)
print(f"R² Score on test set: {r2:.4f}")

# Save the scaler so you can reuse it for new predictions
scaler_path = f"scaler_{SYMBOL}.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to {scaler_path}")

R² Score on test set: 0.3862
Scaler saved to scaler_AAPL.pkl
